# Figure 4: Batch integration

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


## Figure 4 | Batch Integration

In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("../../data/hca_nuclei.h5ad")
# Match the exported IDs: training can remove unexpressed genes.
adata = adata[
    np.load("../../results/hca_nuclei/scLDM_no_batch/scLDM_cell_names.npy"),
    np.load("../../results/hca_nuclei/scLDM_no_batch/scLDM_gene_names.npy"),
].copy()


In [ ]:
import scanpy as sc

_, n_cells = sc.pp.filter_genes(adata, min_cells=1, inplace=False)
adata.var["n_cells"] = n_cells

In [ ]:
# Load scLDM latent representations
z_cells_scldm = np.load('../../results/hca_nuclei/scLDM_no_batch/scLDM_cell_latent.npy')
z_genes_scldm = np.load('../../results/hca_nuclei/scLDM_no_batch/scLDM_gene_latent.npy')

if z_cells_scldm.shape[1] != z_genes_scldm.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm.shape}, genes={z_genes_scldm.shape}")

adata.obsm['scLDM'] = z_cells_scldm
adata.varm['scLDM'] = z_genes_scldm


# Load scLDM batch latent representations
z_cells_scldm_batch = np.load('../../results/hca_nuclei/scLDM_batch_full_full/scLDM_cell_latent.npy')
z_genes_scldm_batch = np.load('../../results/hca_nuclei/scLDM_batch_full_full/scLDM_gene_latent.npy')

if z_cells_scldm_batch.shape[1] != z_genes_scldm_batch.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm_batch.shape}, genes={z_genes_scldm_batch.shape}")

adata.obsm['scLDM_batch'] = z_cells_scldm_batch
adata.varm['scLDM_batch'] = z_genes_scldm_batch

# Load scLDM batch latent representations
z_cells_scldm_batch = np.load('../../results/hca_nuclei/scLDM_batch_full_full_2D/scLDM_cell_latent.npy')
z_genes_scldm_batch = np.load('../../results/hca_nuclei/scLDM_batch_full_full_2D/scLDM_gene_latent.npy')

if z_cells_scldm_batch.shape[1] != z_genes_scldm_batch.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm_batch.shape}, genes={z_genes_scldm_batch.shape}")

adata.obsm['scLDM_batch_2D'] = z_cells_scldm_batch
adata.varm['scLDM_batch_2D'] = z_genes_scldm_batch

z_cells_scldm_batch = np.load('../../results/hca_nuclei/scLDM_no_batch_2D/scLDM_cell_latent.npy')

adata.obsm['scLDM_no_batch_2D'] = z_cells_scldm_batch



In [ ]:
# Load scVI latent representations
z_cells_pca = np.load('../../results/hca_nuclei/PCA/X_pca_vanilla_cell_latent.npy')

adata.obsm['pca'] = z_cells_pca

In [ ]:
# Load scVI latent representations
z_cells_scvi = np.load('../../results/hca_nuclei/scVI/scVI_cell_latent.npy')

adata.obsm['scVI'] = z_cells_scvi

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scLDM")
sc.tl.umap(adata, random_state=42, key_added="umap_scLDM")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scLDM_batch")
sc.tl.umap(adata, random_state=42, key_added="umap_scLDM_batch")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scVI")
sc.tl.umap(adata, random_state=42, key_added="umap_scVI")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="pca")
sc.tl.umap(adata, random_state=42, key_added="umap_pca")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# Extract coordinates
umap = adata.obsm["scLDM_batch_2D"]
keys_to_plot = ["cell_type", "cell_source", 'donor']

for key in keys_to_plot:
    celltypes = adata.obs[key]

    # Build a palette for cell types
    categories = pd.Categorical(celltypes).categories

    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    palette = dict(zip(categories, colors))

    fig, ax = plt.subplots(figsize=(1.5, 1.20))

    # Plot umap by cell type
    for ct in categories:
        idx = (celltypes == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/hca_umap_scldm_batch_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# Extract coordinates
umap = adata.obsm["scLDM_no_batch_2D"]
keys_to_plot = ["cell_type", "cell_source", 'donor']

for key in keys_to_plot:
    celltypes = adata.obs[key]

    # Build a palette for cell types
    categories = pd.Categorical(celltypes).categories

    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    palette = dict(zip(categories, colors))

    fig, ax = plt.subplots(figsize=(1.5, 1.20))

    # Plot umap by cell type
    for ct in categories:
        idx = (celltypes == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/hca_umap_scldm_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_cluster_legend_pdf(
    categories,
    palette,
    output_pdf,
    ncol=1,
    fontsize=6,
    marker_size=4.0,
    columnspacing=0.8,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2,
):
    """
    Save a standalone legend-only PDF for cluster colors.

    Parameters
    ----------
    categories : list[str]
        Ordered category names.
    palette : dict
        Mapping {category: color}.
    output_pdf : str
        Output path ending in .pdf.
    ncol : int
        Number of legend columns.
    fontsize : float
        Legend text size in pt. Nature-style target: ~5–7 pt.
    marker_size : float
        Marker size in pt for legend keys.
    """

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    handles = [
        Line2D(
            [0], [0],
            linestyle="None",
            marker="o",
            markersize=marker_size,
            markerfacecolor=palette[cat],
            markeredgecolor=palette[cat],
            markeredgewidth=0.0,
            label=str(cat),
        )
        for cat in categories
    ]

    # Rough figure size estimate so the legend lays out predictably.
    n_items = len(categories)
    n_rows = math.ceil(n_items / ncol)
    fig_w = max(1.2, 1.15 * ncol + 0.55 * ncol)
    fig_h = max(0.35, 0.22 * n_rows + 0.18)

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles=handles,
        labels=categories,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.8,
        handletextpad=handletextpad,
        columnspacing=columnspacing,
        labelspacing=labelspacing,
        borderpad=borderpad,
        markerscale=1.0,
    )

    fig.savefig(
        output_pdf,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )
    plt.close(fig)

In [ ]:
# categories = list(adata.obs["cell_type"].cat.categories) + ["Gene"]

# palette_legend = palette.copy()
# palette_legend["Gene"] = "#7A1E1E"

# save_cluster_legend_pdf(
#     categories=categories,
#     palette=palette_legend,
#     output_pdf="output/fig_4/hca_cell_type_legend.svg",
#     ncol=1,
#     fontsize=6,
#     marker_size=4.0,
# )

## Joint representation

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

# ----- 1.  Build a placeholder expression matrix for the genes -----
# Gene rows have zero-filled expression values; their embeddings are stored separately.
n_cells, n_genes = adata.n_obs, adata.n_vars
if sp.issparse(adata.X):
    placeholder_X = sp.csr_matrix((n_genes, n_genes), dtype=adata.X.dtype)
else:
    placeholder_X = np.zeros((n_genes, n_genes), dtype=adata.X.dtype)

# ----- 2.  Stack the real cells with the pseudo-cells (genes) -----
X_combined = (
    sp.vstack([adata.X, placeholder_X], format="csr")
    if sp.issparse(adata.X)
    else np.vstack([adata.X, placeholder_X])
)

# ----- 3.  Create an .obs that labels each row as cell / gene -----
obs_combined = pd.concat(
    [
        adata.obs.assign(entity="cell"),               # keep existing cell metadata
        pd.DataFrame({"entity": "gene"}, index=adata.var_names)  # one row per gene
    ]
)

# ----- 4.  Assemble the new AnnData object -----
adata_combo = sc.AnnData(
    X=X_combined,
    obs=obs_combined,
    var=adata.var.copy()            # keep original gene metadata as .var
)

# ----- 5.  Concatenate the embeddings and store in .obsm -----

adata_combo.obsm["scLDM"] = np.vstack([
    adata.obsm["scLDM"],      # cells (n_cells × dim)
    adata.varm["scLDM"]       # genes (n_genes × dim)
])

adata_combo.obsm["scLDM_batch"] = np.vstack([
    adata.obsm["scLDM_batch"],      # cells (n_cells × dim)
    adata.varm["scLDM_batch"]       # genes (n_genes × dim)
])

adata_combo.obsm["scLDM_batch_2D"] = np.vstack([
    adata.obsm["scLDM_batch_2D"],      # cells (n_cells × dim)
    adata.varm["scLDM_batch_2D"]       # genes (n_genes × dim)
])

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata_combo, use_rep="scLDM_batch", n_neighbors=30)
sc.tl.umap(adata_combo, random_state=42, key_added="umap_scLDM_batch")

In [ ]:
celltypes = adata_combo.obs["cell_type"]
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette_celltypes = dict(zip(categories, colors))



cell_source_groups = adata_combo.obs["cell_source"]
categories = pd.Categorical(cell_source_groups).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette_cell_source_groups = dict(zip(categories, colors))


donor_groups = adata_combo.obs["donor"]
categories = pd.Categorical(donor_groups).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette_donor_groups = dict(zip(categories, colors))



In [ ]:
categories = list(adata_combo.obs["cell_type"].cat.categories) + ["Gene"]


rename_map = {
    "Ventricular Cardiomyocyte": "Ventricular CM",
    "Atrial Cardiomyocyte": "Atrial CM",
    "Smooth muscle cells": "SMC",
    "Ventricular_Cardiomyocyte": "Ventricular CM",
    "Atrial_Cardiomyocyte": "Atrial CM",
    "Smooth_muscle_cells": "SMC",
}

categories = [rename_map.get(c, c) for c in categories]
palette = {rename_map.get(k, k): v for k, v in palette_celltypes.items()}


palette["Gene"] = "#7A1E1E"


save_cluster_legend_pdf(
    categories=categories,
    palette=palette,
    output_pdf="output/fig_4/hca_cell_type_legend.svg",
    ncol=1,
    fontsize=6,
    marker_size=4.0,
)

In [ ]:
categories = list(adata_combo.obs["cell_source"].cat.categories)


rename_map = {
    "Harvard-Nuclei": "Harvard",
    "Sanger-Nuclei": "Sanger",
}

categories = [rename_map.get(c, c) for c in categories]
palette = {rename_map.get(k, k): v for k, v in palette_cell_source_groups.items()}

save_cluster_legend_pdf(
    categories=categories,
    palette=palette,
    output_pdf="output/fig_4/hca_cell_source_legend.svg",
    ncol=1,
    fontsize=6,
    marker_size=4.0,
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import numpy as np
import pandas as pd
from adjustText import adjust_text
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# -----------------------------
# Data
# -----------------------------
umap = np.asarray(adata_combo.obsm["umap_scLDM_batch"], dtype=float)

cell_mask = (adata_combo.obs["entity"] == "cell").to_numpy()
gene_mask = (adata_combo.obs["entity"] == "gene").to_numpy()

genes_xy = umap[gene_mask]
gene_names = adata_combo.obs_names[gene_mask].to_numpy()  # should match adata_combo.var_names

# Gene colors from n_counts in .var
if "n_counts" not in adata_combo.var.columns:
    raise ValueError("Expected 'n_counts' in adata_combo.var")

gene_n_counts = pd.to_numeric(
    adata_combo.var.reindex(gene_names)["n_cells"], errors="coerce"
).to_numpy(dtype=float)

gene_color = np.log1p(np.clip(gene_n_counts, a_min=0, a_max=None))
finite = np.isfinite(gene_color)
if not finite.any():
    raise ValueError("No finite n_counts values found for genes.")
vmin, vmax = np.nanpercentile(gene_color[finite], [1, 99])

# -----------------------------
# Helpers
# -----------------------------
def build_palette(categories):
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return dict(zip(categories, colors))

# -----------------------------
# Plot
# -----------------------------
labels = ["cell_type"]
rng = np.random.default_rng(42)  # reproducible random draw order

for label in labels:
    # categories from CELLS only
    cell_labels_series = adata_combo.obs.loc[cell_mask, label].astype(str)
    categories = list(pd.Categorical(cell_labels_series).categories)

    if label == "cell_type" and "palette_celltypes" in globals():
        palette = palette_celltypes
    else:
        palette = build_palette(categories)

    fig, ax = plt.subplots(figsize=(4.5, 4.5))

    # Randomize category plotting order (reduces systematic overplot bias)
    categories_plot = categories.copy()
    rng.shuffle(categories_plot)

    label_all = adata_combo.obs[label].astype(str).to_numpy()

    # Cells by category
    for ct in categories_plot:
        idx = cell_mask & (label_all == ct)
        point_ids = np.flatnonzero(idx)
        if point_ids.size == 0:
            continue
        point_ids = rng.permutation(point_ids)  # random within category
        ax.scatter(
            umap[point_ids, 0],
            umap[point_ids, 1],
            s=1,
            c=[palette[ct]],
            alpha=1,
            linewidths=0,
            label=ct,
            rasterized=True
        )

    # Genes colored by n_counts
    gene_sc = ax.scatter(
        genes_xy[:, 0],
        genes_xy[:, 1],
        s=1,
        c="#7A1E1E",
        #cmap="viridis",
        #vmin=vmin,
        #vmax=vmax,
        alpha=1,
        linewidths=0,
        label="genes",
        rasterized=True
    )

    #cbar = fig.colorbar(gene_sc, ax=ax, fraction=0.03, pad=0.01)
    #cbar.set_label("log1p(n_counts)", fontsize=6)
    #cbar.ax.tick_params(labelsize=5, length=2)

    if label == "cell_type":
        markers = [
            "MYL2", "MYH7", "IRX4",
            "RGS5", "CSPG4", "ABCC9",
            "DCN", "LUM", "COL1A1",
            "PECAM1", "VWF", "EMCN",
            "NPPA", "MYH6", "SLN",
            "LST1", "TYROBP", "FCER1G",
            "ACTA2", "TAGLN", "MYH11",
            "CD3D", "IL7R", "LTB",
            "ADIPOQ", "PLIN1", "FABP4",
            "SNAP25", "RBFOX3", "TUBB3",
            "WT1", "MSLN", "KRT19",
        ]
        selected_genes = np.array(markers)
        sel = np.isin(gene_names, selected_genes)
        genes_sel = genes_xy[sel]
        gene_names_sel = gene_names[sel]

        texts = []
        for (x, y), name in zip(genes_sel, gene_names_sel):
            texts.append(
                ax.text(
                    x, y, name,
                    fontsize=5,
                    color="black",
                    alpha=1,
                    zorder=6,
                    bbox=dict(
                        boxstyle="round,pad=0.15",
                        facecolor="white",
                        edgecolor="none",
                        alpha=0.5
                    )
                )
            )

        adjust_text(
            texts,
            ax=ax,
            only_move={"points": "xy", "text": "xy"},
            expand_points=(3.0, 3.0),
            expand_text=(1.3, 1.5),
            force_points=1.2,
            force_text=0.35,
            force_pull=0.01,
            lim=800,
            arrowprops=dict(arrowstyle="-", color="black", lw=0.25, alpha=0.6),
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(
        f"output/fig_4/hca_joint_umap_latent_{label}.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600
    )
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import numpy as np
import pandas as pd
from adjustText import adjust_text
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# -----------------------------
# Data
# -----------------------------
umap = np.asarray(adata_combo.obsm["scLDM_batch_2D"], dtype=float)

cell_mask = (adata_combo.obs["entity"] == "cell").to_numpy()
gene_mask = (adata_combo.obs["entity"] == "gene").to_numpy()

genes_xy = umap[gene_mask]
gene_names = adata_combo.obs_names[gene_mask].to_numpy()  # should match adata_combo.var_names

# Gene colors from n_counts in .var
if "n_counts" not in adata_combo.var.columns:
    raise ValueError("Expected 'n_counts' in adata_combo.var")

gene_n_counts = pd.to_numeric(
    adata_combo.var.reindex(gene_names)["n_cells"], errors="coerce"
).to_numpy(dtype=float)

gene_color = np.log1p(np.clip(gene_n_counts, a_min=0, a_max=None))
finite = np.isfinite(gene_color)
if not finite.any():
    raise ValueError("No finite n_counts values found for genes.")
vmin, vmax = np.nanpercentile(gene_color[finite], [1, 99])

# -----------------------------
# Helpers
# -----------------------------
def build_palette(categories):
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return dict(zip(categories, colors))

# -----------------------------
# Plot
# -----------------------------
labels = ["cell_type"]
rng = np.random.default_rng(42)  # reproducible random draw order

for label in labels:
    # categories from CELLS only
    cell_labels_series = adata_combo.obs.loc[cell_mask, label].astype(str)
    categories = list(pd.Categorical(cell_labels_series).categories)

    if label == "cell_type" and "palette_celltypes" in globals():
        palette = palette_celltypes
    else:
        palette = build_palette(categories)

    fig, ax = plt.subplots(figsize=(4.5, 4.5))

    # Randomize category plotting order (reduces systematic overplot bias)
    categories_plot = categories.copy()
    rng.shuffle(categories_plot)

    label_all = adata_combo.obs[label].astype(str).to_numpy()

    # Genes colored by n_counts
    gene_sc = ax.scatter(
        genes_xy[:, 0],
        genes_xy[:, 1],
        s=1,
        c="#7A1E1E",
        #cmap="viridis",
        #vmin=vmin,
        #vmax=vmax,
        alpha=0.5,
        linewidths=0,
        label="genes",
        rasterized=True
    )


    # Cells by category
    for ct in categories_plot:
        idx = cell_mask & (label_all == ct)
        point_ids = np.flatnonzero(idx)
        if point_ids.size == 0:
            continue
        point_ids = rng.permutation(point_ids)  # random within category
        ax.scatter(
            umap[point_ids, 0],
            umap[point_ids, 1],
            s=1,
            c=[palette[ct]],
            alpha=1,
            linewidths=0,
            label=ct,
            rasterized=True
        )


    #cbar = fig.colorbar(gene_sc, ax=ax, fraction=0.03, pad=0.01)
    #cbar.set_label("log1p(n_counts)", fontsize=6)
    #cbar.ax.tick_params(labelsize=5, length=2)

if label == "cell_type":
    markers = [
        "FCER1G", "TYROBP", "LST1",
        "CD3D", "IL7R", "LTB",
        "PECAM1", "VWF", "EMCN",
        "COL1A1", "LUM", "DCN",
        "ADIPOQ", "PLIN1", "FABP4",
        "NPPA", "MYH6", "SLN",
        "MYH7", "MYL2", "IRX4",
        "WT1", "MSLN", "KRT19",
    ]

    selected_genes = np.array(markers)
    sel = np.isin(gene_names, selected_genes)
    genes_sel = genes_xy[sel]
    gene_names_sel = gene_names[sel]

    # redraw selected genes on top so the anchor point is explicit
    ax.scatter(
        genes_sel[:, 0],
        genes_sel[:, 1],
        s=10,
        c="#7A1E1E",
        edgecolors="white",
        linewidths=0.4,
        zorder=7,
        rasterized=True,
    )

    texts = []
    for (x, y), name in zip(genes_sel, gene_names_sel):
        texts.append(
            ax.text(
                x, y, name,
                fontsize=5,
                color="black",
                zorder=8,
                bbox=dict(
                    boxstyle="round,pad=0.12",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.75,
                ),
            )
        )

    adjust_text(
        texts,
        ax=ax,
        only_move={"points": "xy", "text": "xy"},
        expand_points=(3.0, 3.0),
        expand_text=(1.2, 1.4),
        force_points=1.0,
        force_text=0.3,
        force_pull=0.05,
        lim=800,
        arrowprops=dict(
            arrowstyle="-",
            color="black",
            lw=0.35,
            alpha=0.8,
        ),
    )


        # adjust_text(
        #     texts,
        #     ax=ax,
        #     only_move={"points": "xy", "text": "xy"},
        #     expand_points=(3.0, 3.0),
        #     expand_text=(1.3, 1.5),
        #     force_points=1.2,
        #     force_text=0.35,
        #     force_pull=0.01,
        #     lim=800,
        #     arrowprops=dict(arrowstyle="-", color="black", lw=0.25, alpha=0.6),
        # # )
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(
        f"output/fig_4/hca_joint_2D_latent_{label}.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600
    )
    plt.show()


## 3D

In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("../../data/hca_nuclei.h5ad")

# Match the exported IDs: training can remove unexpressed genes.
adata = adata[
    np.load("../../results/hca_nuclei/scLDM_batch_full_full_3D/scLDM_cell_names.npy"),
    np.load("../../results/hca_nuclei/scLDM_batch_full_full_3D/scLDM_gene_names.npy"),
].copy()

z_cells_3d = np.load('../../results/hca_nuclei/scLDM_batch_full_full_3D/scLDM_cell_latent.npy')
z_genes_3d = np.load('../../results/hca_nuclei/scLDM_batch_full_full_3D/scLDM_gene_latent.npy')

if z_cells_3d.shape[1] != z_genes_3d.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_3d.shape}, genes={z_genes_3d.shape}")

adata.obsm['scLDM_3d'] = z_cells_3d
adata.varm['scLDM_3d'] = z_genes_3d

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal


def _build_palette_for_series(series):
    categories = list(pd.Categorical(series.astype(str)).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


COMMON_CATEGORIES = {}
COMMON_PALETTES = {}
for _label in ["cell_type", "label"]:
    if _label in adata.obs.columns:
        _cats, _pal = _build_palette_for_series(adata.obs[_label])
        COMMON_CATEGORIES[_label] = _cats
        COMMON_PALETTES[_label] = _pal



In [ ]:

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

Z_cells = np.asarray(adata.obsm["scLDM_3d"], dtype=float)
Z_genes = np.asarray(adata.varm["scLDM_3d"], dtype=float)

if Z_cells.shape[1] != 3 or Z_genes.shape[1] != 3:
    raise ValueError(f"Expected 3D embeddings. Got cells={Z_cells.shape}, genes={Z_genes.shape}")

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]

outdir = Path("output/fig_4")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        fig = plt.figure(figsize=(2, 2))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=0.3, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.scatter(
            Z_genes[:, 0], Z_genes[:, 1], Z_genes[:, 2],
            s=0.1, c="#7A1E1E", alpha=.5, linewidths=0, depthshade=False, rasterized=True 
        )

        ax.view_init(elev=elev, azim=azim)
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_zlim(-3, 3)
        
        ax.set_position([0.02, 0.02, 0.96, 0.97])
        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        plt.savefig(
            outdir / f"hca_scLDM_3d_{label}_{view_name}.png",
            bbox_inches="tight",
            pad_inches=0,
            dpi=600,
        )
        plt.show()


## Labels

In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 5,
    "svg.fonttype": "none",   # keep text editable
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

genes = [
    "ADIPOQ", "CD3D", "COL1A1", "DCN", "EMCN", "FABP4", "FCER1G",
    "IL7R", "IRX4", "KRT19", "LST1", "LTB", "LUM", "MSLN", "MYH6",
    "MYH7", "MYL2", "NPPA", "PECAM1", "PLIN1", "SLN", "TYROBP",
    "VWF", "WT1"
]

# grid layout
n_cols = 4
n_rows = math.ceil(len(genes) / n_cols)

# figure size in inches
fig_w = 3.35
row_height = 0.22
fig_h = max(1.2, n_rows * row_height)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.axis("off")

for i, gene in enumerate(genes):
    row = n_rows - 1 - (i // n_cols)
    col = i % n_cols

    x = col + 0.5
    y = row + 0.5

    ax.text(
        x, y, gene,
        ha="center",
        va="center",
        fontsize=5,
        bbox=dict(
            facecolor="white",
            alpha=0.75,
            edgecolor="none",
            boxstyle="round,pad=0.18",
        ),
    )

outpath = Path("output/fig_4/gene_labels_grid.svg")
fig.savefig(
    outpath,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.02,
    transparent=True,
)
plt.show()
plt.close(fig)

print(f"Saved to: {outpath}")